# 01 — Data Understanding
### 2023/24 Kenya Housing Survey · MSc Dissertation
**Strathmore University · Data Science & Analytics**

This notebook performs a thorough, multi-angle data understanding pass across all relevant KHS microdata files.
The output directly informs feature engineering decisions in `02_feature_engineering.ipynb`.

**Sections:**
1. Environment setup
2. File inventory & sizes
3. Household file — deep profile
4. Dwelling Units — deep profile
5. Individual Microdata — deep profile
6. Financial files (Mortgage + Loan)
7. County Microdata
8. Cross-file join audit
9. Risk dimension feasibility assessment
10. Feature engineering decision log

---
## 1. Environment Setup

In [ ]:
# Mount Drive and set up repo
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content')
!git clone https://github.com/YOUR_USERNAME/khs-housing-dissertation.git 2>/dev/null || \
    (cd khs-housing-dissertation && git pull)
os.chdir('khs-housing-dissertation')
sys.path.insert(0, 'src')

In [ ]:
!pip install -q polars pyarrow matplotlib seaborn scipy

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy import stats

warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────
DRIVE   = Path("/content/drive/MyDrive/KHS_Dissertation")
RAW     = DRIVE / "data" / "raw"
PQ      = DRIVE / "data" / "parquet"
OUT     = DRIVE / "outputs"
FIGS    = OUT / "figures"
TABLES  = OUT / "tables"
for p in [FIGS, TABLES]: p.mkdir(parents=True, exist_ok=True)

# ── Plot style ───────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F8F6',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': '500',
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'font.family': 'sans-serif',
})
TEAL   = '#00695C'
PURPLE = '#6A1B9A'
AMBER  = '#E65100'
RED    = '#B71C1C'
BLUE   = '#1565C0'
GRAY   = '#546E7A'

# ── Value label loader ───────────────────────────────────────────────
with open(PQ / "household_variable_labels.json") as f:
    VAR_LABELS = json.load(f)
with open(PQ / "household_value_labels.json") as f:
    VAL_LABELS = json.load(f)

def label(col):
    """Return human-readable question text for a column code."""
    return VAR_LABELS.get(col, col)

def decode(col, series):
    """Map numeric codes to string labels using value label dict."""
    mapping = VAL_LABELS.get(col.upper(), VAL_LABELS.get(col, {}))
    return series.map(lambda x: mapping.get(str(int(x)), str(x)) if pd.notna(x) else np.nan)

print("Environment ready.")

---
## 2. File Inventory

In [ ]:
FILES = {
    'household':    'Household_Information_Data.parquet',
    'dwelling':     'Dwelling_Units_Microdata.parquet',
    'individual':   'Individual_Microdata.parquet',
    'mortgage':     'Housing_Mortgage.parquet',
    'loan':         'Housing_Loan_Microdata.parquet',
    'county':       'County_Microdata.parquet',
    'real_estate':  'Real_Estate_Microdata.parquet',
    'land':         'Land_Parcels_Microdata.parquet',
    'financiers':   'Housing_Financiers_Microdata.parquet',
    'water':        'Water_Service_Providers_Microdata.parquet',
    'nema':         'NEMA_Microdata.parquet',
    'housing_types':'Type_of_Housing_Units_Microdata.parquet',
    'project':      'Project_Information_Microdata.parquet',
    'institution':  'Institution_Microdata.parquet',
}

print(f"{'File':<20} {'Rows':>10} {'Cols':>6} {'Size MB':>10} {'Status'}")
print("-" * 60)
inventory = {}
for key, fname in FILES.items():
    path = PQ / fname
    if path.exists():
        df_tmp = pl.read_parquet(path)
        size_mb = path.stat().st_size / 1e6
        inventory[key] = {'rows': df_tmp.shape[0], 'cols': df_tmp.shape[1], 'mb': size_mb}
        print(f"{key:<20} {df_tmp.shape[0]:>10,} {df_tmp.shape[1]:>6} {size_mb:>10.2f}  ✓")
        del df_tmp
    else:
        inventory[key] = None
        print(f"{key:<20} {'—':>10} {'—':>6} {'—':>10}  ✗ not found")

---
## 3. Household File — Deep Profile
The spine of the entire study. Every other file joins to this.

In [ ]:
hh = pl.read_parquet(PQ / FILES['household'])
print(f"Household file: {hh.shape[0]:,} rows × {hh.shape[1]} columns")

In [ ]:
# ── 3.1 Full null audit ──────────────────────────────────────────────
null_audit = (
    hh.null_count()
      .unpivot(variable_name='column', value_name='nulls')
      .with_columns((pl.col('nulls') / len(hh) * 100).round(1).alias('pct_null'))
      .sort('nulls', descending=True)
)

# Classify columns by missingness tier
tiers = {
    'complete   (0%)':        null_audit.filter(pl.col('pct_null') == 0).shape[0],
    'sparse     (1–20%)':     null_audit.filter(pl.col('pct_null').is_between(1, 20)).shape[0],
    'moderate   (21–50%)':    null_audit.filter(pl.col('pct_null').is_between(21, 50)).shape[0],
    'high       (51–90%)':    null_audit.filter(pl.col('pct_null').is_between(51, 90)).shape[0],
    'structural (91–100%)':   null_audit.filter(pl.col('pct_null') > 90).shape[0],
}
print("\nNull distribution across 392 columns:")
for tier, count in tiers.items():
    bar = '█' * int(count / 3)
    print(f"  {tier}  {count:>4} cols  {bar}")

# Show columns we'll actually use (< 60% null)
usable = null_audit.filter(pl.col('pct_null') < 60)
print(f"\nUsable columns (< 60% null): {usable.shape[0]} of {hh.shape[1]}")

In [ ]:
# ── 3.2 Geography: county × urban/rural distribution ────────────────
hh_pd = hh.to_pandas()

# Decode urban/rural
hh_pd['residence'] = hh_pd['a07_1'].map({1: 'Rural', 2: 'Urban'})

county_counts = (
    hh_pd.groupby(['a01', 'residence'])
         .size()
         .unstack(fill_value=0)
         .assign(total=lambda x: x.sum(axis=1))
         .sort_values('total', ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 7))
county_counts[['Rural', 'Urban']].plot(
    kind='bar', stacked=True, ax=ax,
    color=[TEAL, PURPLE], width=0.8, edgecolor='none'
)
ax.set_title('Sample distribution by county and residence type')
ax.set_xlabel('County')
ax.set_ylabel('Number of households')
ax.legend(['Rural', 'Urban'])
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(FIGS / '01_county_sample_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nUrban households: {(hh_pd['residence']=='Urban').sum():,} ({(hh_pd['residence']=='Urban').mean()*100:.1f}%)")
print(f"Rural households: {(hh_pd['residence']=='Rural').sum():,} ({(hh_pd['residence']=='Rural').mean()*100:.1f}%)")

In [ ]:
# ── 3.3 Financial stress columns — core for HFVS ────────────────────
fin_cols = {
    'c14_1': 'Monthly expenditure (KES)',
    'c14_2': 'Monthly savings (KES)',
    'c14_3': 'Monthly investments (KES)',
    'k05':   'Monthly rent paid (KES)',
    'k25':   'Willing/able to spend on housing (KES)',
    'k27':   'Loan accessible now (KES)',
    'l12':   'Amount used to purchase/construct dwelling (KES)',
    'l14':   'Estimated current value of dwelling (KES)',
    'l15':   'Estimated monthly rental value (KES)',
}

print("Financial column summary:\n")
print(f"{'Column':<8} {'Label':<42} {'Non-null':>8} {'%':>6} {'Median':>12} {'Mean':>12} {'Max':>14}")
print("-" * 106)

for col, lbl in fin_cols.items():
    if col not in hh_pd.columns:
        print(f"{col:<8} {lbl:<42} {'—':>8}")
        continue
    s = pd.to_numeric(hh_pd[col], errors='coerce')
    nn = s.notna().sum()
    pct = nn / len(s) * 100
    med = s.median() if nn > 0 else np.nan
    mean = s.mean() if nn > 0 else np.nan
    mx = s.max() if nn > 0 else np.nan
    print(f"{col:<8} {lbl:<42} {nn:>8,} {pct:>5.1f}% {med:>12,.0f} {mean:>12,.0f} {mx:>14,.0f}")

In [ ]:
# ── 3.4 Rent burden ratio — the most critical derived variable ───────
hh_pd['expenditure'] = pd.to_numeric(hh_pd['c14_1'], errors='coerce')
hh_pd['monthly_rent'] = pd.to_numeric(hh_pd['k05'], errors='coerce')

# Only renters have k05 — that is structurally expected
renters = hh_pd[hh_pd['monthly_rent'].notna() & hh_pd['expenditure'].notna()].copy()
renters = renters[(renters['monthly_rent'] > 0) & (renters['expenditure'] > 0)]

# Winsorise at 1st and 99th percentile
p1_rent, p99_rent = renters['monthly_rent'].quantile([0.01, 0.99])
p1_exp, p99_exp   = renters['expenditure'].quantile([0.01, 0.99])
renters = renters[
    renters['monthly_rent'].between(p1_rent, p99_rent) &
    renters['expenditure'].between(p1_exp, p99_exp)
]

renters['rent_burden'] = renters['monthly_rent'] / renters['expenditure']
renters['rent_burden'] = renters['rent_burden'].clip(0, 1)

print(f"Households with both rent AND expenditure data: {len(renters):,}")
print(f"  — of total households: {len(renters)/len(hh_pd)*100:.1f}%")
print(f"\nRent burden distribution:")
print(renters['rent_burden'].describe().round(3))

print(f"\nRent-burdened (> 30% of expenditure): "
      f"{(renters['rent_burden'] > 0.30).mean()*100:.1f}% of renters")
print(f"Severely burdened (> 50%):             "
      f"{(renters['rent_burden'] > 0.50).mean()*100:.1f}% of renters")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution
axes[0].hist(renters['rent_burden'], bins=50, color=TEAL, edgecolor='none', alpha=0.85)
axes[0].axvline(0.30, color=AMBER, linewidth=1.5, linestyle='--', label='30% threshold')
axes[0].axvline(0.50, color=RED,   linewidth=1.5, linestyle='--', label='50% threshold')
axes[0].set_title('Rent burden ratio distribution (renters)')
axes[0].set_xlabel('Rent / Monthly expenditure')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=9)

# By residence type
for label_name, color in [('Rural', TEAL), ('Urban', PURPLE)]:
    subset = renters[renters['residence'] == label_name]['rent_burden']
    axes[1].hist(subset, bins=40, alpha=0.6, label=label_name, color=color, edgecolor='none')
axes[1].set_title('Rent burden by residence type')
axes[1].set_xlabel('Rent / Monthly expenditure')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGS / '02_rent_burden_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.5 Tenure security columns ──────────────────────────────────────
tenure_cols = {
    'i00':  'Ownership of land',
    'k01':  'To whom rent is paid',
    'k02':  'Written tenancy agreement',
    'k29':  'Ever had rent dispute',
    'k34':  'Demolitions in neighbourhood (5 yrs)',
    'k35':  'Threat of household eviction',
    'l26':  'Threat of dwelling being demolished',
    'l29':  'Has development approvals',
}

print("Tenure security columns:\n")
print(f"{'Column':<8} {'Label':<45} {'Non-null':>8} {'%':>6}")
print("-" * 75)
for col, lbl in tenure_cols.items():
    if col not in hh_pd.columns:
        print(f"{col:<8} {lbl:<45} {'not found':>8}")
        continue
    nn = hh_pd[col].notna().sum()
    pct = nn / len(hh_pd) * 100
    top = hh_pd[col].value_counts().head(3).to_dict()
    print(f"{col:<8} {lbl:<45} {nn:>8,} {pct:>5.1f}%  top: {top}")

In [ ]:
# ── 3.6 Physical hazard columns (enumerator-observed) ────────────────
hazard_main = {
    'e06': 'Locality prone to flooding',
    'e07': 'Locality prone to mudslides',
    'e08': 'Land terrain of the dwelling',
}
proximity_cols = {f'e09__{i}': v for i, v in enumerate([
    'River/Lake', 'Swamp', 'Wetland', 'Quarry', 'Dumpsite',
    'Factory/Industry', 'Forest', 'Bars/Nightlife', 'Worship centre',
    'Airport', 'Busy road', 'Helipad', 'Ocean'
], start=1)}

all_hazard = {**hazard_main, **proximity_cols}

print("Physical hazard columns (enumerator-observed — high quality):\n")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Flood/mudslide prevalence
flood_data  = hh_pd['e06'].value_counts(normalize=True) * 100
mud_data    = hh_pd['e07'].value_counts(normalize=True) * 100

hazard_summary = pd.DataFrame({'Flood prone': flood_data, 'Mudslide prone': mud_data}).fillna(0)
hazard_summary.plot(kind='bar', ax=axes[0], color=[TEAL, AMBER], width=0.6, edgecolor='none')
axes[0].set_title('Flood and mudslide exposure (%)')
axes[0].set_ylabel('% of households')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Proximity hazards
prox_pct = {}
for col, lbl in proximity_cols.items():
    if col in hh_pd.columns:
        pct = hh_pd[col].mean() * 100 if hh_pd[col].max() <= 1 else \
              (hh_pd[col] == 1).mean() * 100
        prox_pct[lbl] = pct

prox_series = pd.Series(prox_pct).sort_values(ascending=True)
prox_series.plot(kind='barh', ax=axes[1], color=PURPLE, edgecolor='none', alpha=0.85)
axes[1].set_title('Proximity hazard prevalence (% of households)')
axes[1].set_xlabel('% of households')

plt.tight_layout()
plt.savefig(FIGS / '03_hazard_prevalence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Flood-prone localities:    {(hh_pd['e06']==1).mean()*100:.1f}% of households")
print(f"Mudslide-prone localities: {(hh_pd['e07']==1).mean()*100:.1f}% of households")

In [ ]:
# ── 3.7 Utility deprivation ──────────────────────────────────────────
# Water
water_labels = VAL_LABELS.get('C01_1', VAL_LABELS.get('c01_1', {}))
water_counts = hh_pd['c01_1'].value_counts()

# Electricity
elec_pct = (hh_pd['c08'] == 1).mean() * 100

# Cooking fuel
cooking_labels = VAL_LABELS.get('C11', VAL_LABELS.get('c11', {}))
cooking_counts = hh_pd['c11'].value_counts()

# Toilet
toilet_labels = VAL_LABELS.get('C04', VAL_LABELS.get('c04', {}))
toilet_counts = hh_pd['c04'].value_counts()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Utility access and deprivation', fontsize=14, fontweight='500', y=1.01)

# Water source
top_water = water_counts.head(8)
top_water_labels = [water_labels.get(str(k), str(k))[:30] for k in top_water.index]
axes[0,0].barh(top_water_labels, top_water.values, color=BLUE, edgecolor='none', alpha=0.85)
axes[0,0].set_title('Main drinking water source')
axes[0,0].set_xlabel('Households')

# Electricity
elec_vals = hh_pd['c08'].value_counts()
elec_labs = {1: 'Connected', 2: 'Not connected'}
elec_labels_list = [elec_labs.get(k, str(k)) for k in elec_vals.index]
axes[0,1].bar(elec_labels_list, elec_vals.values, color=[TEAL, GRAY], edgecolor='none', alpha=0.85)
axes[0,1].set_title('Electricity (main grid) connection')
axes[0,1].set_ylabel('Households')

# Cooking fuel
top_cook = cooking_counts.head(7)
top_cook_labels = [cooking_labels.get(str(k), str(k))[:30] for k in top_cook.index]
axes[1,0].barh(top_cook_labels, top_cook.values, color=AMBER, edgecolor='none', alpha=0.85)
axes[1,0].set_title('Main cooking fuel')
axes[1,0].set_xlabel('Households')

# Toilet
top_toilet = toilet_counts.head(7)
top_toilet_labels = [toilet_labels.get(str(k), str(k))[:30] for k in top_toilet.index]
axes[1,1].barh(top_toilet_labels, top_toilet.values, color=RED, edgecolor='none', alpha=0.85)
axes[1,1].set_title('Main toilet facility type')
axes[1,1].set_xlabel('Households')

plt.tight_layout()
plt.savefig(FIGS / '04_utility_deprivation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.8 Housing barriers — d20 block ────────────────────────────────
barrier_cols = {
    'd20__1': 'High cost of rent',
    'd20__2': 'High cost of land',
    'd20__3': 'Lack of legal ownership docs',
    'd20__4': "Don't qualify for a loan",
    'd20__5': 'High cost of materials',
    'd20__6': 'High interest rates',
    'd20__7': 'Houses too costly to buy',
    'd20__8': 'Low income',
    'd20__9': 'Proximity to amenities',
    'd20__10': 'Not interested in moving',
}

barrier_pct = {}
for col, lbl in barrier_cols.items():
    if col in hh_pd.columns:
        barrier_pct[lbl] = (hh_pd[col] == 1).mean() * 100

barrier_series = pd.Series(barrier_pct).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [RED if v > 30 else AMBER if v > 15 else TEAL for v in barrier_series.values]
ax.barh(barrier_series.index, barrier_series.values, color=colors, edgecolor='none', alpha=0.88)
ax.set_title('Housing barriers — % of households reporting each barrier')
ax.set_xlabel('% of all households')
ax.axvline(30, color='gray', linewidth=0.8, linestyle='--')
for i, v in enumerate(barrier_series.values):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=8.5)
plt.tight_layout()
plt.savefig(FIGS / '05_housing_barriers.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey finding: top barriers are candidates for financial exclusion sub-score.")

In [ ]:
# ── 3.9 Survey weights ───────────────────────────────────────────────
print("Survey weights (hhweight):")
print(hh_pd['hhweight'].describe().round(3))
print(f"\nTotal weighted population: {hh_pd['hhweight'].sum():,.0f}")
print("NOTE: All analysis from notebook 03 onward must use hhweight.")
print("Unweighted EDA here is fine for feature engineering decisions.")

---
## 4. Dwelling Units — Deep Profile

In [ ]:
# Load dwelling units
dwell_path = PQ / FILES['dwelling']
if dwell_path.exists():
    dwell_pd = pl.read_parquet(dwell_path).to_pandas()
    print(f"Dwelling Units: {dwell_pd.shape[0]:,} rows × {dwell_pd.shape[1]} columns")
    print(f"\nColumns: {list(dwell_pd.columns)}")
else:
    print("Dwelling Units file not found — convert DTA first.")
    dwell_pd = None

In [ ]:
if dwell_pd is not None:
    # ── 4.1 Extract dwelling codebook ────────────────────────────────
    import pandas as pd
    try:
        dwell_reader = pd.io.stata.StataReader(str(RAW / FILES['dwelling'].replace('.parquet', '.dta')))
        dwell_var_labels = dwell_reader.variable_labels()
        dwell_val_labels = dwell_reader.value_labels()

        def stringify_keys(obj):
            if isinstance(obj, dict):
                return {str(k): stringify_keys(v) for k, v in obj.items()}
            return obj

        with open(PQ / 'dwelling_variable_labels.json', 'w') as f:
            json.dump(dwell_var_labels, f, indent=2)
        with open(PQ / 'dwelling_value_labels.json', 'w') as f:
            json.dump(stringify_keys(dwell_val_labels), f, indent=2)

        print("Dwelling codebook extracted. Variable labels:")
        for col, lbl in dwell_var_labels.items():
            print(f"  {col:<25} → {lbl}")
    except Exception as e:
        print(f"Could not read DTA for codebook: {e}")
        print("Continuing with column names only.")
        dwell_var_labels = {c: c for c in dwell_pd.columns}

In [ ]:
if dwell_pd is not None:
    # ── 4.2 Null audit on dwelling file ──────────────────────────────
    dwell_null = dwell_pd.isnull().mean() * 100
    print("Null rates in dwelling file:")
    print(dwell_null.sort_values(ascending=False).round(1).to_string())

    # ── 4.3 Find construction material columns ────────────────────────
    material_keywords = ['wall', 'floor', 'roof', 'ceil', 'material', 'construct', 'struct']
    mat_cols = [
        col for col in dwell_pd.columns
        if any(k in col.lower() for k in material_keywords)
        or any(k in dwell_var_labels.get(col, '').lower() for k in material_keywords)
    ]
    print(f"\nConstruction material columns found: {mat_cols}")
    for col in mat_cols:
        print(f"\n{col} — {dwell_var_labels.get(col, '')}")
        print(dwell_pd[col].value_counts().head(8))

In [ ]:
if dwell_pd is not None:
    # ── 4.4 Room count and overcrowding ──────────────────────────────
    room_keywords = ['room', 'bedroom', 'sleep']
    room_cols = [
        col for col in dwell_pd.columns
        if any(k in col.lower() for k in room_keywords)
        or any(k in dwell_var_labels.get(col, '').lower() for k in room_keywords)
    ]
    print(f"Room/bedroom columns: {room_cols}")

    if room_cols:
        for col in room_cols[:3]:
            r = pd.to_numeric(dwell_pd[col], errors='coerce')
            print(f"\n{col}: mean={r.mean():.2f}, median={r.median():.1f}, max={r.max()}")

---
## 5. Individual Microdata — Deep Profile

In [ ]:
ind_path = PQ / FILES['individual']
if ind_path.exists():
    # Sample 30% for EDA — full file is 60 MB
    ind_pl = pl.read_parquet(ind_path).sample(fraction=0.30, seed=42)
    ind_pd = ind_pl.to_pandas()
    print(f"Individual file sample: {len(ind_pd):,} rows × {ind_pd.shape[1]} columns")
    print(f"(30% sample for EDA; full file loaded in feature engineering)")
    print(f"\nColumns: {list(ind_pd.columns[:30])} ...")
else:
    print("Individual file not found.")
    ind_pd = None

In [ ]:
if ind_pd is not None:
    # Extract individual codebook
    try:
        ind_reader = pd.io.stata.StataReader(str(RAW / FILES['individual'].replace('.parquet','.dta')))
        ind_var_labels = ind_reader.variable_labels()

        def stringify_keys(obj):
            if isinstance(obj, dict):
                return {str(k): stringify_keys(v) for k, v in obj.items()}
            return obj

        ind_val_labels = stringify_keys(ind_reader.value_labels())

        with open(PQ / 'individual_variable_labels.json', 'w') as f:
            json.dump(ind_var_labels, f, indent=2)
        with open(PQ / 'individual_value_labels.json', 'w') as f:
            json.dump(ind_val_labels, f, indent=2)

        print("Individual codebook extracted.\n")

        # Find education, employment, age, income columns
        econ_keywords = ['edu', 'employ', 'work', 'income', 'age', 'sex', 'gender',
                         'occupation', 'salary', 'wage', 'earn']
        econ_cols = [
            (col, lbl) for col, lbl in ind_var_labels.items()
            if any(k in col.lower() or k in lbl.lower() for k in econ_keywords)
        ]
        print("Economic/demographic columns in Individual file:")
        for col, lbl in econ_cols:
            nn = ind_pd[col].notna().sum() if col in ind_pd.columns else 0
            print(f"  {col:<25} → {lbl[:60]}  [{nn:,} non-null]")
    except Exception as e:
        print(f"Could not extract codebook: {e}")
        ind_var_labels = {c: c for c in ind_pd.columns}

In [ ]:
if ind_pd is not None:
    # Households per individual record
    hh_size = ind_pd.groupby('interview__key').size()
    print("Individuals per household:")
    print(hh_size.describe().round(2))

    fig, ax = plt.subplots(figsize=(9, 4))
    hh_size.clip(1, 15).value_counts().sort_index().plot(
        kind='bar', ax=ax, color=PURPLE, edgecolor='none', alpha=0.85
    )
    ax.set_title('Household size distribution (from Individual file, 30% sample)')
    ax.set_xlabel('Number of individuals in household')
    ax.set_ylabel('Count of households')
    plt.tight_layout()
    plt.savefig(FIGS / '06_household_size.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 6. Financial Files — Mortgage & Loan

In [ ]:
mort_path = PQ / FILES['mortgage']
loan_path = PQ / FILES['loan']

if mort_path.exists():
    mort_pd = pl.read_parquet(mort_path).to_pandas()
    print(f"Mortgage file: {mort_pd.shape[0]:,} rows × {mort_pd.shape[1]} cols")
    print(mort_pd.head(3))
else:
    mort_pd = None
    print("Mortgage file not found.")

if loan_path.exists():
    loan_pd = pl.read_parquet(loan_path).to_pandas()
    print(f"\nLoan file: {loan_pd.shape[0]:,} rows × {loan_pd.shape[1]} cols")
    print(loan_pd.head(3))
else:
    loan_pd = None
    print("Loan file not found.")

In [ ]:
# ── Join rate check ──────────────────────────────────────────────────
hh_keys = set(hh_pd['interview__key'].unique())
print(f"Total household keys: {len(hh_keys):,}")

if mort_pd is not None:
    mort_keys = set(mort_pd['interview__key'].unique())
    overlap = len(hh_keys & mort_keys)
    print(f"Mortgage file unique keys: {len(mort_keys):,}")
    print(f"  Match to household: {overlap:,} ({overlap/len(hh_keys)*100:.1f}% of households)")

if loan_pd is not None:
    loan_keys = set(loan_pd['interview__key'].unique())
    overlap_l = len(hh_keys & loan_keys)
    print(f"Loan file unique keys: {len(loan_keys):,}")
    print(f"  Match to household: {overlap_l:,} ({overlap_l/len(hh_keys)*100:.1f}% of households)")
    print("\nNOTE: Low match rate = low formal finance uptake = this IS the finding.")

---
## 7. County Microdata

In [ ]:
county_path = PQ / FILES['county']
if county_path.exists():
    county_pd = pl.read_parquet(county_path).to_pandas()
    print(f"County file: {county_pd.shape[0]:,} rows × {county_pd.shape[1]} cols")
    print(f"\nColumns: {list(county_pd.columns)}")
    print("\nFirst 5 rows:")
    print(county_pd.head())
else:
    print("County file not found.")
    county_pd = None

---
## 8. Cross-File Join Audit

In [ ]:
# Comprehensive join rate report
print("=" * 65)
print("JOIN RATE AUDIT — all files vs household spine")
print("=" * 65)
print(f"{'File':<20} {'HH keys matched':>18} {'% coverage':>12} {'Join key'}")
print("-" * 65)

files_to_check = [
    ('dwelling',   PQ / FILES.get('dwelling',''),   'interview__key'),
    ('individual', PQ / FILES.get('individual',''), 'interview__key'),
    ('mortgage',   PQ / FILES.get('mortgage',''),   'interview__key'),
    ('loan',       PQ / FILES.get('loan',''),        'interview__key'),
    ('county',     PQ / FILES.get('county',''),      'countycode'),
]

join_results = {}
for name, path, key in files_to_check:
    if not path or not path.exists():
        print(f"{name:<20} {'file not found':>18}")
        continue
    df_tmp = pl.read_parquet(path)
    if key not in df_tmp.columns or key not in hh_pd.columns:
        print(f"{name:<20} {'key not found':>18}")
        continue
    other_keys = set(df_tmp[key].to_list())
    spine_keys = set(hh_pd[key].tolist())
    matched = len(spine_keys & other_keys)
    pct = matched / len(spine_keys) * 100
    join_results[name] = pct
    status = '✓ safe' if pct > 70 else '△ partial' if pct > 20 else '✗ sparse'
    print(f"{name:<20} {matched:>15,}   {pct:>9.1f}%   {key}  {status}")
    del df_tmp

print("\nDecision rule: files with > 70% coverage → direct join.")
print("Files with < 30% → flag variable only (1=matched, 0=not).")

---
## 9. Risk Dimension Feasibility Assessment

In [ ]:
# Systematic check of every planned HFVS component
print("HOUSING FINANCIAL VULNERABILITY SCORE — component feasibility\n")
print("=" * 72)

components = {
    'DIMENSION 1: Financial stress': {
        'c14_1':  ('Monthly expenditure', 'core — income proxy'),
        'k05':    ('Monthly rent paid',    'core — rent burden numerator'),
        'c14_2':  ('Monthly savings',      'secondary'),
        'd20__8': ('Low income (flag)',     'binary supplement'),
        'd20__4': ('No loan access (flag)', 'binary supplement'),
    },
    'DIMENSION 2: Tenure insecurity': {
        'k35':  ('Eviction threat',        'core'),
        'i00':  ('Owns land',              'core'),
        'k02':  ('Written tenancy agreement','core'),
        'k29':  ('Had rent dispute',        'secondary'),
        'k34':  ('Demolitions in area',     'secondary'),
        'l26':  ('Demolition threat',       'owners only'),
    },
    'DIMENSION 3: Physical hazard': {
        'e06':     ('Flood prone locality',   'core — enumerator-observed'),
        'e07':     ('Mudslide prone locality', 'core — enumerator-observed'),
        'e09__1':  ('Near river/lake',         'secondary'),
        'e09__2':  ('Near swamp',              'secondary'),
        'e09__5':  ('Near dumpsite',           'secondary'),
        'e09__6':  ('Near factory',            'secondary'),
        'e09__11': ('Near busy road',          'secondary'),
    },
    'DIMENSION 4: Dwelling quality (from Dwelling Units file)': {
        'wall_material':  ('Wall construction type',  'pending — check dwelling file'),
        'floor_material': ('Floor construction type', 'pending — check dwelling file'),
        'roof_material':  ('Roof construction type',  'pending — check dwelling file'),
        'num_rooms':      ('Number of rooms',         'pending — check dwelling file'),
    },
    'DIMENSION 5: Utility deprivation': {
        'c08':    ('No main grid electricity', 'core'),
        'c01_1':  ('Unimproved water source',  'core'),
        'c04':    ('Unimproved toilet',         'core'),
        'c11':    ('Solid fuel for cooking',    'secondary'),
        'c07':    ('No handwashing facility',   'secondary'),
    },
}

for dim_name, cols in components.items():
    print(f"\n{dim_name}")
    print("-" * 60)
    for col, (description, note) in cols.items():
        if col in hh_pd.columns:
            nn = hh_pd[col].notna().sum()
            pct = nn / len(hh_pd) * 100
            status = '✓' if pct > 70 else '△' if pct > 30 else '✗'
            print(f"  {status} {col:<15} {description:<35} {pct:>5.1f}% non-null  [{note}]")
        elif 'pending' in note:
            print(f"  ? {col:<15} {description:<35} pending dwelling file")
        else:
            print(f"  ✗ {col:<15} {'column not found':<35}")

In [ ]:
# ── Correlation between available financial columns ──────────────────
numeric_fin_cols = ['c14_1', 'c14_2', 'c14_3', 'k05', 'k25']
fin_sub = hh_pd[[c for c in numeric_fin_cols if c in hh_pd.columns]].apply(
    pd.to_numeric, errors='coerce'
)
fin_sub = fin_sub.dropna(how='all')

corr = fin_sub.corr(method='spearman')

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1, ax=ax,
    xticklabels=[label(c)[:20] for c in corr.columns],
    yticklabels=[label(c)[:20] for c in corr.columns],
    linewidths=0.5, square=True
)
ax.set_title('Spearman correlation — financial columns')
plt.tight_layout()
plt.savefig(FIGS / '07_financial_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── County-level hazard and burden heatmap ───────────────────────────
county_summary = hh_pd.groupby('a01').agg(
    n_households     = ('interview__key', 'count'),
    pct_urban        = ('a07_1', lambda x: (x == 2).mean() * 100),
    pct_flood        = ('e06', lambda x: (x == 1).mean() * 100),
    pct_mudslide     = ('e07', lambda x: (x == 1).mean() * 100),
    pct_elec         = ('c08', lambda x: (x == 1).mean() * 100),
    pct_eviction     = ('k35', lambda x: (x == 1).mean() * 100),
    pct_rent_dispute = ('k29', lambda x: (x == 1).mean() * 100),
    mean_expenditure = ('c14_1', lambda x: pd.to_numeric(x, errors='coerce').median()),
    mean_rent        = ('k05',   lambda x: pd.to_numeric(x, errors='coerce').median()),
).reset_index().sort_values('pct_flood', ascending=False)

# Normalise for heatmap display
heatmap_cols = ['pct_flood', 'pct_mudslide', 'pct_eviction', 'pct_rent_dispute']
heatmap_data = county_summary.set_index('a01')[heatmap_cols]
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

fig, ax = plt.subplots(figsize=(9, 14))
sns.heatmap(
    heatmap_norm, annot=heatmap_data.round(1), fmt='.1f',
    cmap='YlOrRd', ax=ax, linewidths=0.3,
    cbar_kws={'label': 'Normalised score'},
    xticklabels=['Flood %', 'Mudslide %', 'Eviction %', 'Rent dispute %']
)
ax.set_title('County-level risk indicator heatmap\n(sorted by flood exposure)', pad=12)
ax.set_ylabel('County')
plt.tight_layout()
plt.savefig(FIGS / '08_county_risk_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Save as table
county_summary.to_csv(TABLES / 'county_risk_summary.csv', index=False)
print("County summary saved to outputs/tables/county_risk_summary.csv")

---
## 10. Feature Engineering Decision Log

In [ ]:
# Print final decisions to inform notebook 02
decisions = """
╔══════════════════════════════════════════════════════════════════════╗
║         FEATURE ENGINEERING DECISIONS — from data understanding     ║
╚══════════════════════════════════════════════════════════════════════╝

TARGET VARIABLE
  - HFVS = composite Housing Financial Vulnerability Score (0–1)
  - Binary version: hfvs_high = 1 if HFVS > 0.60
  - Computed from weighted sum of 5 dimension scores

DIMENSION 1: Financial stress  (weight: TBD via PCA/EFA)
  ✓ rent_burden_ratio = k05 / c14_1  [only for renters]
  ✓ rent_stressed     = 1 if rent_burden > 0.30
  ✓ savings_flag      = 1 if c14_2 == 0 or null
  ✓ low_income_flag   = d20__8
  NOTE: c14_1 has high coverage — use as primary income proxy
        Winsorise at 1st/99th percentile before use

DIMENSION 2: Tenure insecurity
  ✓ no_land_ownership = 1 if i00 != 'owns'
  ✓ no_written_lease  = 1 if k02 == 0 (renters)
  ✓ eviction_threat   = k35
  ✓ rent_dispute_hist = k29
  ✓ demo_threat       = k34 (neighbourhood demolitions)
  tenure_score = sum of above flags (0–5)

DIMENSION 3: Physical hazard
  ✓ flood_zone      = e06  [enumerator-observed — high trust]
  ✓ mudslide_zone   = e07  [enumerator-observed — high trust]
  ✓ hazard_count    = sum of e09__1 through e09__13
  ✓ high_risk_prox  = 1 if near dumpsite OR factory OR swamp

DIMENSION 4: Dwelling quality  (from Dwelling Units file)
  ⚠  PENDING — depends on dwelling codebook
  Planned: wall_durable, floor_durable, roof_durable (binary)
  Planned: overcrowding = persons / rooms > 3

DIMENSION 5: Utility deprivation
  ✓ no_electricity = 1 if c08 != 1
  ✓ unsafe_water   = 1 if c01_1 in [unimproved sources]
  ✓ poor_sanitation= 1 if c04 in [unimproved toilet types]
  ✓ solid_fuel     = 1 if c11 in [wood, charcoal, kerosene]
  deprivation_index = sum of above (0–4)

STRUCTURAL MISSINGNESS DECISIONS
  k-section (k01–k39): renters only — run dimension 1 & 2 separately
                        for renters vs owners, then combine
  l-section (l01–l32): owners only — same approach
  High-null cols (>90%): drop from main model, flag-only in supplementary

MODELLING DECISIONS
  - All weighted analyses MUST use hhweight
  - GroupKFold CV by county (prevents spatial leakage)
  - Imputation: median for continuous, mode for categorical
  - Scale: StandardScaler on continuous features before logistic baseline
  - XGBoost handles missing natively — preferred primary model

DIMENSION WEIGHTING STRATEGY
  Option A: Equal weights (0.20 each) — simple, defensible
  Option B: PCA-derived weights — data-driven, methodologically stronger
  Option C: Expert weights from actuarial literature — most publishable
  → Recommend: run all three, compare county rankings, report concordance
"""
print(decisions)

# Save decisions as text file
with open(TABLES / 'feature_engineering_decisions.txt', 'w') as f:
    f.write(decisions)
print("Decision log saved to outputs/tables/feature_engineering_decisions.txt")

---
## Push to GitHub

In [ ]:
# Configure git (first time only)
!git config user.email "your_email@example.com"
!git config user.name "Your Name"

# Stage notebook and outputs (not data)
!git add notebooks/01_data_understanding.ipynb
!git add outputs/figures/ outputs/tables/ 2>/dev/null || true
!git status

In [ ]:
!git commit -m "feat: complete data understanding — EDA, null audit, risk dimension feasibility"
!git push origin main